# Lecture 2 — Discrete Diffusion Language Models
**IISc DS 207 · Apr 7, 2026 · Apoorv Saxena (Inception AI)**

---

In Lecture 1 we explored **autoregressive** decoding: one token at a time, left to right.

Today: **What if we could generate all tokens at once and refine iteratively?**

In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
import time
import numpy as np

---
## Part 1 — BERT Can Fill in Blanks

BERT is a **masked language model**: given a sentence with some tokens replaced by `[MASK]`,
it predicts what goes in the blanks.

Let's first understand step by step what BERT does under the hood.

In [2]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-large")
bert = AutoModelForMaskedLM.from_pretrained("answerdotai/ModernBERT-large")
bert.eval()
print(f"BERT loaded — {sum(p.numel() for p in bert.parameters())/1e6:.0f}M parameters")
print(f"Vocabulary size: {tokenizer.vocab_size}")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 173/173 [00:00<00:00, 32757.64it/s]

BERT loaded — 396M parameters
Vocabulary size: 50280


### Step 1: Tokenization

First, we convert text into token IDs. BERT uses WordPiece tokenization.

In [10]:
text = "The capital of India is [MASK]."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs["input_ids"]

print(f"Text:      {text}")
print(f"Token IDs: {input_ids[0].tolist()}")
print(f"Tokens:    {tokenizer.convert_ids_to_tokens(input_ids[0])}")
print(f"\n[MASK] token ID = {tokenizer.mask_token_id}")

Text:      The capital of India is [MASK].
Token IDs: [50281, 510, 5347, 273, 5427, 310, 50284, 15, 50282]
Tokens:    ['[CLS]', 'The', 'Ġcapital', 'Ġof', 'ĠIndia', 'Ġis', '[MASK]', '.', '[SEP]']

[MASK] token ID = 50284


### Step 2: Forward pass

We feed the token IDs into BERT. The output is a **logit** for every token in the vocabulary, at every position.

In [11]:
with torch.no_grad():
    output = bert(**inputs)
    logits = output.logits

print(f"Input shape:  {input_ids.shape}        # (batch=1, seq_len={input_ids.shape[1]})")
print(f"Output shape: {logits.shape}  # (batch=1, seq_len={logits.shape[1]}, vocab_size={logits.shape[2]})")
print(f"\nFor EVERY position, BERT outputs a score for EVERY word in the vocabulary.")

Input shape:  torch.Size([1, 9])        # (batch=1, seq_len=9)
Output shape: torch.Size([1, 9, 50368])  # (batch=1, seq_len=9, vocab_size=50368)

For EVERY position, BERT outputs a score for EVERY word in the vocabulary.


In [12]:
logits.shape

torch.Size([1, 9, 50368])

### Step 3: Get predictions for the [MASK] position

We find the `[MASK]` position, then look at BERT's predicted probabilities there.

In [13]:
mask_pos = (input_ids[0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
mask_pos

6

In [14]:
# Find the [MASK] position
print(f"[MASK] is at position {mask_pos}")

# Get logits at that position → convert to probabilities via softmax
mask_logits = logits[0, mask_pos]          # shape: (vocab_size,)
mask_probs = F.softmax(mask_logits, dim=-1)

# Top 5 predictions
top_probs, top_ids = mask_probs.topk(5)
print(f"\nTop 5 predictions for [MASK]:")
for prob, tid in zip(top_probs, top_ids):
    print(f"  {tokenizer.decode(tid):15s}  {prob:.1%}")

[MASK] is at position 6

Top 5 predictions for [MASK]:
   Delhi           74.4%
   Mumbai          24.6%
   India           0.5%
   New             0.2%
   London          0.1%


### Wrap it into a reusable function

In [15]:
def bert_fill_masks(text, top_k=3):
    """Show BERT's top-k predictions for each [MASK] in the text."""
    inputs = tokenizer(text, return_tensors="pt")
    mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        logits = bert(**inputs).logits

    filled_ids = inputs["input_ids"][0].clone()

    print(f"Input:  {text}\n")
    for pos in mask_positions:
        probs = F.softmax(logits[0, pos], dim=-1)
        top_probs, top_ids = probs.topk(top_k)
        filled_ids[pos] = top_ids[0]
        tokens = [tokenizer.decode(tid) for tid in top_ids]
        preds = ", ".join(f"{t} ({p:.1%})" for t, p in zip(tokens, top_probs))
        print(f"  [MASK] at position {pos.item()}: {preds}")

    print(f"\nTop-1 filled: {tokenizer.decode(filled_ids, skip_special_tokens=True)}")

### One mask — easy!

In [16]:
bert_fill_masks("The capital of France is")

Input:  The capital of France is [MASK].

  [MASK] at position 6:  Paris (98.5%),  Nice (0.7%),  Lyon (0.5%)

Top-1 filled: The capital of France is Paris.


### Two masks — still pretty good

In [17]:
bert_fill_masks("The [MASK] of [MASK] is Paris.")

Input:  The [MASK] of [MASK] is Paris.

  [MASK] at position 2:  capital (45.1%),  city (24.0%),  Capital (5.5%)
  [MASK] at position 4:  France (26.2%),  choice (6.7%),  Paris (5.1%)

Top-1 filled: The capital of France is Paris.


### What about ALL masks?

In [20]:
bert_fill_masks("[MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK]")

Input:  [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK]

  [MASK] at position 1: # (21.9%), What (3.0%), The (2.7%)
  [MASK] at position 2:  is (6.4%), : (5.1%),  the (2.0%)
  [MASK] at position 3: : (6.0%), 
 (5.8%),  is (2.6%)
  [MASK] at position 4: 
 (10.2%), : (3.9%),  the (1.6%)
  [MASK] at position 5: 
 (12.6%), : (3.2%), . (2.0%)
  [MASK] at position 6: 
 (11.1%), . (2.3%), - (2.0%)
  [MASK] at position 7: . (9.5%), 
 (2.9%), : (2.2%)
  [MASK] at position 8: 
 (53.6%), 

 (6.5%), . (5.6%)

Top-1 filled: # is:


.



**Observation:** With 100% masks, BERT gives **garbage**.

**Why?** BERT was trained with only ~15% of tokens masked. 100% masking is out of distribution.

→ What if we trained a model to handle **any** amount of masking — from 0% to 100%?

→ **This is the core idea of discrete diffusion.** (We'll come back to this.)

But first — can we use BERT to generate text like GPT does?

---
## Part 2 — AR Generation with BERT

In Lecture 1, we saw GPT generate text **left to right**: predict one token, append it, repeat.

Can BERT do the same? Let's try: put `[MASK]` at the end, predict it, append, repeat.

In [21]:
def bert_ar_generate(prompt, max_new_tokens=20, verbose=True):
    """
    Pseudo-autoregressive generation using BERT:
    Append many [MASK] tokens at the end.
    At each step, predict only the next mask, then replace it with the prediction.
    """
    tokens = tokenizer.encode(prompt, add_special_tokens=False)
    generated = []

    if verbose:
        print(f"Prompt: {prompt}\n")

    for step in range(max_new_tokens):
        # [CLS] + prompt tokens + generated tokens + remaining masks + [SEP]
        num_remaining_masks = max_new_tokens - step
        input_ids = torch.tensor([[
            tokenizer.cls_token_id
        ] + tokens + generated + [tokenizer.mask_token_id] * num_remaining_masks + [
            tokenizer.sep_token_id
        ]])

        with torch.no_grad():
            logits = bert(input_ids).logits

        # Predict only the first remaining [MASK]
        mask_pos = 1 + len(tokens) + len(generated)
        probs = F.softmax(logits[0, mask_pos], dim=-1)
        next_token = probs.argmax().item()

        if next_token == tokenizer.sep_token_id:
            break

        generated.append(next_token)

        if verbose:
            print(f"  Step {step+1}: {tokenizer.decode(tokens + generated)}")

    result = tokenizer.decode(tokens + generated)
    if verbose:
        print(f"\nFinal: {result}")
    return result

In [22]:
bert_ar_generate("the meaning of life is", max_new_tokens=15)

Prompt: the meaning of life is

  Step 1: the meaning of life is to
  Step 2: the meaning of life is to be
  Step 3: the meaning of life is to be happy
  Step 4: the meaning of life is to be happy

  Step 5: the meaning of life is to be happy


  Step 6: the meaning of life is to be happy

What
  Step 7: the meaning of life is to be happy

What is
  Step 8: the meaning of life is to be happy

What is the
  Step 9: the meaning of life is to be happy

What is the meaning
  Step 10: the meaning of life is to be happy

What is the meaning of
  Step 11: the meaning of life is to be happy

What is the meaning of life
  Step 12: the meaning of life is to be happy

What is the meaning of life?
  Step 13: the meaning of life is to be happy

What is the meaning of life?

  Step 14: the meaning of life is to be happy

What is the meaning of life?
be
  Step 15: the meaning of life is to be happy

What is the meaning of life?
be happy

Final: the meaning of life is to be happy

What is the meaning 

'the meaning of life is to be happy\n\nWhat is the meaning of life?\nbe happy'

In [23]:
bert_ar_generate("the capital of india", max_new_tokens=5)

Prompt: the capital of india

  Step 1: the capital of india is
  Step 2: the capital of india is the
  Step 3: the capital of india is the ind
  Step 4: the capital of india is the indian
  Step 5: the capital of india is the indian capital

Final: the capital of india is the indian capital


'the capital of india is the indian capital'

**Observation:** BERT *can* do AR generation, but it's not great — it was trained to fill in
**middle** tokens, not predict the **next** token.

This highlights complementary strengths:
- **GPT**: great left-to-right, can't fill in the middle
- **BERT**: can fill blanks anywhere, but awkward at left-to-right

Now let's try something different: instead of going left to right,
what if we reveal tokens in **any order** — most confident first?

---
## Part 3 — Iterative Unmasking (Diffusion-Style Generation)

Here's the core idea behind discrete diffusion inference:
1. Start with all `[MASK]` tokens (plus a fixed prefix for context)
2. Run the model → get predictions for every masked position
3. Unmask the **most confident** predictions
4. Repeat until all tokens are revealed

In [24]:
def iterative_unmask(prefix, gen_length=10, num_steps=5, verbose=True):
    """
    Discrete-diffusion-style generation using BERT.
    Given a fixed prefix, generate gen_length new tokens by iterative unmasking.
    Each step: predict all masked positions, unmask the most confident ones.
    """
    mask_id = tokenizer.mask_token_id
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    
    # [CLS] + prefix + [MASK]*gen_length + [SEP]
    gen_start = 1 + len(prefix_ids)
    gen_end = gen_start + gen_length
    input_ids = torch.tensor([
        [tokenizer.cls_token_id] + prefix_ids + [mask_id]*gen_length + [tokenizer.sep_token_id]
    ])
    
    def display_gen():
        parts = []
        for i in range(gen_start, gen_end):
            if input_ids[0, i] == mask_id:
                parts.append('____')
            else:
                parts.append(tokenizer.decode(input_ids[0, i]))
        return ' '.join(parts)
    
    if verbose:
        print(f"Prefix: {prefix}")
        print(f"Generating {gen_length} tokens in {num_steps} steps\n")
        print(f"  Start:  {prefix} {display_gen()}")
    
    for step in range(num_steps):
        with torch.no_grad():
            logits = bert(input_ids).logits
        
        gen_tokens = input_ids[0, gen_start:gen_end]
        gen_logits = logits[0, gen_start:gen_end]
        
        masked = (gen_tokens == mask_id)
        if not masked.any():
            break
        
        probs = F.softmax(gen_logits, dim=-1)
        best_probs, best_tokens = probs.max(dim=-1)
        
        n_masked = masked.sum().item()
        n_to_unmask = max(1, n_masked // max(1, num_steps - step))
        
        confidences = best_probs.clone()
        confidences[~masked] = -1
        _, top_indices = confidences.topk(min(n_to_unmask, n_masked))
        
        for idx in top_indices:
            gen_tokens[idx] = best_tokens[idx]
        input_ids[0, gen_start:gen_end] = gen_tokens
        
        if verbose:
            print(f"  Step {step+1}: {prefix} {display_gen()}")
    
    result = tokenizer.decode(input_ids[0, gen_start:gen_end])
    if verbose:
        print(f"\nFinal: {prefix} {result}")
    return result

In [27]:
iterative_unmask("the capital of india", gen_length=10, num_steps=2)

Prefix: the capital of india
Generating 10 tokens in 2 steps

  Start:  the capital of india ____ ____ ____ ____ ____ ____ ____ ____ ____ ____
  Step 1: the capital of india 
 
 ____ ____ 
 
 ____ ____ ____ 

  Step 2: the capital of india 
 
 #  Delhi 
 
 #  of : 


Final: the capital of india 

# Delhi

# of:



'\n\n# Delhi\n\n# of:\n'

In [30]:
iterative_unmask("machine learning is useful because", gen_length=10, num_steps=6)

Prefix: machine learning is useful because
Generating 10 tokens in 6 steps

  Start:  machine learning is useful because ____ ____ ____ ____ ____ ____ ____ ____ ____ ____
  Step 1: machine learning is useful because  it ____ ____ ____ ____ ____ ____ ____ ____ ____
  Step 2: machine learning is useful because  it ____ ____ ____ ____ ____ ____ ____ ____ 

  Step 3: machine learning is useful because  it  can ____ ____ ____ ____ ____ ____ . 

  Step 4: machine learning is useful because  it  can  be ____ ____ ____ ____  data . 

  Step 5: machine learning is useful because  it  can  be  used  to ____ ____  data . 

  Step 6: machine learning is useful because  it  can  be  used  to  analyze  the  data . 


Final: machine learning is useful because  it can be used to analyze the data.



' it can be used to analyze the data.\n'

### The compute–quality trade-off

With iterative unmasking, **you choose** how many steps to use. More steps = better quality.

In [31]:
prompt = "The capital of India is"
length = 12

for steps in [2, 4, 6, 12]:
    print(f"\n{'='*55}")
    print(f"  {steps} steps ({steps} forward passes for {length} tokens)")
    print(f"{'='*55}")
    iterative_unmask(prompt, gen_length=length, num_steps=steps)


  2 steps (2 forward passes for 12 tokens)
Prefix: The capital of India is
Generating 12 tokens in 2 steps

  Start:  The capital of India is ____ ____ ____ ____ ____ ____ ____ ____ ____ ____ ____ ____
  Step 1: The capital of India is  Delhi 
 
 
 
 ____ ____ ____ ____ ____ ____ .
  Step 2: The capital of India is  Delhi 
 
 
 
 The  capital  of  capital  is  Delhi .

Final: The capital of India is  Delhi



The capital of capital is Delhi.

  4 steps (4 forward passes for 12 tokens)
Prefix: The capital of India is
Generating 12 tokens in 4 steps

  Start:  The capital of India is ____ ____ ____ ____ ____ ____ ____ ____ ____ ____ ____ ____
  Step 1: The capital of India is ____ 
 
 ____ ____ ____ ____ ____ ____ ____ ____ .
  Step 2: The capital of India is : 
 
 The ____  of ____ ____ ____ ____ ____ .
  Step 3: The capital of India is : 
 
 The  capital  of  India  is ____ ____ ____ .
  Step 4: The capital of India is : 
 
 The  capital  of  India  is  Delhi  New  Delhi .

Final: The

**Compare to AR:**
- AR needs **12 forward passes** for 12 tokens (one per token)
- Iterative unmasking with 4 steps: **4 forward passes** for 12 tokens — 3× fewer!

### Infilling — something AR models can't do!

Because we unmask in any order, we can give a **partial** sentence and fill the gaps.
GPT can only extend from the left — this is fundamentally different.

In [32]:
def bert_infill(text_with_blanks, num_steps=5, verbose=True):
    """Fill [MASK] tokens in a partially-specified sentence via iterative unmasking."""
    inputs = tokenizer(text_with_blanks, return_tensors="pt")
    input_ids = inputs["input_ids"].clone()
    mask_id = tokenizer.mask_token_id
    
    if verbose:
        print(f"Input: {text_with_blanks}\n")
    
    for step in range(num_steps):
        masked = (input_ids[0] == mask_id)
        if not masked.any():
            break
        
        with torch.no_grad():
            logits = bert(input_ids).logits
        
        probs = F.softmax(logits[0], dim=-1)
        best_probs, best_tokens = probs.max(dim=-1)
        
        n_masked = masked.sum().item()
        n_to_unmask = max(1, n_masked // max(1, num_steps - step))
        
        confidences = best_probs.clone()
        confidences[~masked] = -1
        _, top_indices = confidences.topk(min(n_to_unmask, n_masked))
        
        for idx in top_indices:
            input_ids[0, idx] = best_tokens[idx]
        
        if verbose:
            display = []
            for i, tid in enumerate(input_ids[0]):
                if tid == mask_id:
                    display.append("____")
                elif tid not in (tokenizer.cls_token_id, tokenizer.sep_token_id):
                    display.append(tokenizer.decode(tid))
            print(f"  Step {step+1}: {' '.join(display)}")
    
    result = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    if verbose:
        print(f"\nFinal: {result}")
    return result

In [35]:
bert_infill("the bat [MASK] [MASK] the fence and [MASK] gracefully", num_steps=3)

Input: the bat [MASK] [MASK] the fence and [MASK] gracefully

  Step 1: the  bat ____ ____  the  fence  and  landed  grace fully
  Step 2: the  bat ____  over  the  fence  and  landed  grace fully
  Step 3: the  bat  flew  over  the  fence  and  landed  grace fully

Final: the bat flew over the fence and landed gracefully


'the bat flew over the fence and landed gracefully'

In [ ]:
bert_infill("[MASK] [MASK] is the capital of [MASK] [MASK]", num_steps=4)

The output is decent but not great. Why?

→ BERT was only trained with ~15% masking. At step 1 (100% masked), BERT is **out of distribution**.

→ We need a model **trained across all noise levels** — that's a **discrete diffusion LLM**!

---
## Part 4 — Visualizing the Forward Process

Diffusion models are trained by **corrupting** clean text with increasing noise.
Let's visualize what the forward (noising) process looks like.

In [ ]:
def visualize_forward_process(text, num_steps=6):
    """Show the forward (noising) process: progressively mask more tokens."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    words = [tokenizer.decode(t) for t in tokens]
    length = len(tokens)
    
    print(f"Forward process: corrupting text ({length} tokens)\n")
    
    for step in range(num_steps + 1):
        t = step / num_steps
        n_mask = int(t * length)
        mask_indices = set(np.random.choice(length, size=n_mask, replace=False)) if n_mask > 0 else set()
        
        display = []
        for i in range(length):
            display.append("[M]" if i in mask_indices else words[i])
        
        pct = f"{t:.0%}".rjust(4)
        print(f"  t={pct} ({n_mask:2d}/{length} masked): {' '.join(display)}")

visualize_forward_process("the quick brown fox jumps over the lazy dog near the river bank")

**Training a diffusion LLM:**
- Sample random noise level t ∈ [0, 1]
- Mask that fraction of tokens
- Train the model to predict the masked tokens given the noisy version

At t=0.15 this is exactly BERT's MLM! The model learns a **spectrum** from easy (few masks) to hard (all masks).

---
## Part 5 — Real Diffusion Language Models

Several models have been trained as proper discrete diffusion LMs:

| Model | Size | Year | Key idea |
|-------|------|------|----------|
| **MDLM** | 200M | 2024 | Clean masked diffusion framework (NeurIPS) |
| **SEDD** | ~200M | 2024 | Score-based discrete diffusion (ICML Best Paper) |
| **LLaDA** | 8B | 2025 | First to rival AR LLMs at scale |
| **Dream** | 7B | 2025 | Instruction-following diffusion LLM |

These use the **exact same iterative unmasking algorithm** we just built —
but produce dramatically better output because they were **trained for this**.

### Running a real diffusion LLM

The smaller models (MDLM, SEDD) require flash_attn (CUDA only).
The larger ones (LLaDA, Dream) need a GPU with 16+ GB VRAM.
Below is code for **Dream** — requires a GPU (e.g. Colab T4).

In [ ]:
# === DREAM DIFFUSION LLM ===
# Requires GPU. On Colab: Runtime → Change runtime type → T4 GPU
# pip install torch transformers==4.46.2

USE_DREAM = False  # Set True on a machine with GPU

if USE_DREAM:
    from transformers import AutoModel, AutoTokenizer as DreamTok
    
    dream_tok = DreamTok.from_pretrained("Dream-org/Dream-v0-Instruct-7B")
    dream = AutoModel.from_pretrained(
        "Dream-org/Dream-v0-Instruct-7B",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    ).to("cuda").eval()
    print("Dream loaded!")
else:
    print("Skipping Dream (no GPU). Pre-computed examples below.")

In [ ]:
if USE_DREAM:
    prompt = "Write a short poem about the ocean."
    messages = [{"role": "user", "content": prompt}]
    text = dream_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = dream_tok(text, return_tensors="pt").to("cuda")
    
    output = dream.diffusion_generate(
        **inputs,
        max_new_tokens=128,
        output_history=True,
        steps=64,
    )
    
    for i, step_ids in enumerate(output.history[::8]):
        decoded = dream_tok.decode(step_ids[0], skip_special_tokens=True)
        print(f"Step {i*8}: {decoded[:120]}...")
    
    final = dream_tok.decode(output.sequences[0], skip_special_tokens=True)
    print(f"\nFinal:\n{final}")

### What real diffusion generation looks like

Here's a real run from Dream (7B diffusion LLM) showing iterative unmasking:

**Prompt:** "Write a short poem about the ocean."

```
Step  0: [M] [M] [M] [M] [M] [M] [M] [M] [M] [M] [M] [M] [M] [M]
Step  8: [M] [M] the [M] [M] of [M] [M] [M] deep [M] [M] [M] [M]
Step 16: beneath the [M] of something deep and [M] [M] blue [M] [M]
Step 24: beneath the weight of something deep and vast , blue [M] [M]
Step 32: beneath the weight of something deep and vast , the blue
         horizon stretches on , [M] and free
Step 48: beneath the weight of something deep and vast , the blue
         horizon stretches on , unbroken and free ,
         the waves roll in with ancient memory
Step 64: Beneath the weight of something deep and vast,
         the blue horizon stretches on, unbroken and free.
         The waves roll in with ancient memory,
         carrying salt and stories from the past.
```

**Observations:**
- Common words ("the", "of", "and") appear first — highest confidence
- Content words fill in later as context builds
- The model refines iteratively — like how humans write!
- 64 forward passes for ~50 tokens (AR would need ~50 sequential passes)

---
## Summary

| | Autoregressive (GPT) | Diffusion (Dream, LLaDA, etc.) |
|---|---|---|
| **Generation order** | Left to right | Any order (most confident first) |
| **Forward passes** | One per token (N) | Configurable steps (T ≪ N) |
| **Infilling** | Needs special training | Built-in! |
| **Quality control** | Temperature, top-k/p | Steps + temperature |
| **Training** | Next-token prediction | Predict masked tokens at all noise levels |

### Key takeaways
1. **Discrete diffusion = BERT MLM at all noise levels** (0% to 100%)
2. **Inference = iterative unmasking** (start from noise, reveal confident tokens first)
3. **Fewer steps = fewer forward passes** → controllable compute–quality trade-off
4. **Infilling is free** — no architectural changes needed

### Models to explore
- **MDLM** (NeurIPS 2024) — clean masked diffusion framework
- **SEDD** (ICML 2024 Best Paper) — score-based discrete diffusion
- **LLaDA** (2025) — diffusion scaled to 8B parameters
- **Dream** (2025) — instruction-following diffusion LLM